# Neu — OCGEO-01 «ОЦЕНЩИК-ГЕОМЕТРИЯ», пилот 1 (Colab)

1. **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить** (Runtime → Change runtime type → T4 GPU).
2. **Среда выполнения → Выполнить все** (Runtime → Run all).
3. Когда Colab попросит доступ к Google Диску — разрешить.

Идёт около 30–40 минут: скачиваются шесть моделей Pythia (около 34 ГБ) и считаются ответы на 2430 задач сложения. Вкладку можно не трогать, но не закрывать.

Результат — папка **Neu_colab/ocgeo_pilot1** на вашем Диске. Когда в последней ячейке появится «готово», напишите Claude «готово». Личных данных ноутбук не пишет.

In [ ]:
# 1. Видеокарта и Google Диск (Colab попросит разрешение)
import os, torch
assert torch.cuda.is_available(), 'Нет видеокарты: Среда выполнения → Сменить среду выполнения → T4 GPU, затем «Выполнить все» заново'
print('видеокарта:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/Neu_colab/ocgeo_pilot1'
os.makedirs(OUT, exist_ok=True)
print('папка результата:', OUT)

In [ ]:
# 2. Код опыта с GitHub — закреплённый коммит, сверка sha256
import hashlib, urllib.request
COMMIT = '46865518a540037bb8da5b5c80cb0e106a7927bb'
BASE = f'https://raw.githubusercontent.com/131ymm-commits/Neu/{COMMIT}/experiments/2026-11-14_OCGEO-01'
SHA = {"ocgeo_extract.py": "e44c2e8bd5193928528f09a4c27e38f189fad500937b008bb279085dcab5416e", "analyze_ocgeo.py": "d7724c385718ec5fade09cf385b69edd730230cd8423b2bc87274a2c9214ffca", "colab_summary.py": "653c722b9a61034d6bc0742998ef7b9740a74f0b43ad81e4f3fbb34f94370c7a"}
for name, path in [('ocgeo_extract.py', 'pilot1/ocgeo_extract.py'), ('analyze_ocgeo.py', 'analyze_ocgeo.py'), ('colab_summary.py', 'pilot1/colab_summary.py')]:
    data = urllib.request.urlopen(f'{BASE}/{path}').read()
    h = hashlib.sha256(data).hexdigest()
    assert h == SHA[name], f'sha256 не совпал: {name}'
    open(name, 'wb').write(data)
    print(name, 'sha256 совпал')

In [ ]:
# 3. Ответы голов и соседи в их представлениях (долго: около 30 минут)
!pip -q install accelerate
!python ocgeo_extract.py

In [ ]:
# 4. Анализ и сжатая сводка
!python colab_summary.py pilot_out pilot_out/summary.txt

In [ ]:
# 5. Запись на Диск
import shutil, glob
for f in glob.glob('pilot_out/*'):
    shutil.copy(f, OUT)
drive.flush_and_unmount()
print('готово: записано в Neu_colab/ocgeo_pilot1 — напишите Claude «готово»')